<a href="https://colab.research.google.com/github/jc020230/practiceai/blob/main/regex_%EA%B3%B5%EB%B6%80%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regular expression excercise

**한글 요약: 이 노트북에서 "아 이래서" 할 포인트 6개**

1. **겹치지 않고 왼쪽부터 찾음.** 한 번 찾은 글자는 다시 안 씀. 그래서 `[ab][ab]`가 `'aba'`에서 `'ab'`만 찾고 `'ba'`는 못 찾음
2. **`re.MULTILINE` 때문에 `^`, `$`가 "줄마다" 적용됨.** 없으면 문자열 전체의 맨 처음/맨 끝만
3. **`*`와 `?`는 "0개"도 되니까 빈 문자열("")이 잔뜩 매칭됨.** `\d*` 결과가 159개나 나오는 이유. 찾을 땐 `+`
4. **`\b`와 `\B`는 숫자 덩어리를 "끝 두 자리"와 "앞쪽"으로 나눠 가짐.** `'1234'` → `\d\d\b`는 `34`, `\d\d\B`는 `12`
5. **괄호가 있으면 Group 줄이 출력됨.** `(?:)`는 묶기만 해서 Group 줄이 안 나옴
6. **`(?:)`는 그룹 번호를 차지하지 않음.** 그래서 마지막 마스킹에서 `\2`가 끝 4자리를 가리킴

**결과의 위치 숫자 읽는 법**: `Match 1 was found at 13-15: aa` = test_str의 13번, 14번 글자 (끝 15는 미포함, 슬라이싱 `test_str[13:15]`와 같음). 줄바꿈 `\n`도 1글자로 셈

각 줄이 시작하는 위치 (이거 알면 위치 숫자가 어느 줄인지 바로 보임):

| 시작 | 줄 | 시작 | 줄 |
| --- | --- | --- | --- |
| 0 | hello world | 62 | il.youp.kwak@gmail.com |
| 12 | 4aa  13 | 85 | 02-111-1234 |
| 20 | aba  b44 | 97 | 010-3234-5555 |
| 29 | ababaa5 | 111 | 01032321111 |
| 37 | yaya--- | 123 | hello send me a message(name.kwak@cau.ac.kr) |
| 45 | ikwak2@cau.ac.kr | 168 | call me (032 111 1424) |

## sample code

In [ ]:
# re = regular expression(정규표현식) 도구. 파이썬에 기본으로 들어있어서 설치 필요 없음
import re


In [ ]:
# 찾을 패턴. 앞의 r은 raw string → 역슬래시(\)를 파이썬이 건드리지 않고 regex에 그대로 넘겨줌 (regex는 항상 r 붙이는 습관)
# "aa" = 글자 a 두 개가 연속으로 붙어 있는 곳
regex = r"aa"

# 연습용 문자열. 괄호 안에 문자열 여러 개를 나란히 쓰면 자동으로 하나로 이어붙여짐 (2장 read_csv URL에서 본 것)
# 각 줄 끝의 \n = 줄바꿈 → 결과적으로 12줄짜리 긴 문자열 하나
# 일부러 다양한 걸 섞어 놓음: 단어, 숫자, aa/ab 반복, 이메일 3개, 전화번호 4개(하이픈/붙여쓰기/공백)
test_str = ("hello world\n"
	"4aa  13\n"
	"aba  b44\n"
	"ababaa5\n"
	"yaya---\n"
	"ikwak2@cau.ac.kr\n"
	"il.youp.kwak@gmail.com\n"
	"02-111-1234\n"
	"010-3234-5555\n"
	"01032321111\n"
	"hello send me a message(name.kwak@cau.ac.kr)\n"
	"call me (032 111 1424)\n")


# re.finditer(패턴, 문자열, 옵션) → 찾은 것들을 "하나씩 꺼낼 수 있는" 형태로 돌려줌 (위치 정보까지 들어있음)
#   re.findall은 글자만 리스트로 주고, finditer는 위치(start, end)까지 줘서 이 노트북에서 씀
# re.MULTILINE → ^(시작)와 $(끝)를 문자열 전체가 아니라 "줄마다" 적용해줘 라는 옵션. Anchors 셀에서 중요해짐
matches = re.finditer(regex, test_str, re.MULTILINE)


In [ ]:
# 찍어보면 <callable_iterator ...> 라고만 나옴 → 리스트가 아니라 "하나씩 꺼내 쓰는 기계(iterator)"라서 내용이 안 보임
# 중요! iterator는 한 번 다 꺼내면 비어버림
#   그래서 아래 셀을 두 번 실행하면 두 번째엔 아무것도 안 나옴 → 위 셀(matches = ...)부터 다시 실행해야 함
#   (여기서 list(matches)를 해도 내용은 보이지만, 그 순간 비어서 아래 for문에서 아무것도 안 나옴)
matches


In [ ]:
# enumerate(matches, start=1) → 찾은 것(match)을 하나씩 꺼내면서 1부터 번호(matchNum)를 붙여줌
for matchNum, match in enumerate(matches, start=1):

    # match.start() = 찾은 곳의 시작 위치 / match.end() = 끝 위치 (끝은 미포함! 13-15면 13, 14번 글자)
    # match.group() = 찾은 글자 자체
    # "...{matchNum}...".format(matchNum=...) → 문자열의 {} 빈칸을 채우는 문법 (f-string의 옛날 버전)
    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    # match.groups() = 패턴 안 괄호( )로 잡은 그룹들. len()으로 그룹 개수
    # 패턴 "aa"에는 괄호가 없어서 0개 → 이 안쪽 for문은 한 번도 안 돌아서 Group 줄이 안 나옴
    # 괄호가 있는 패턴(Groups 셀)에서는 Group 1, 2... 줄이 같이 출력됨
    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1 # 그룹 번호는 1부터라서 +1

        # match.start(n), match.end(n), match.group(n) → n번 그룹의 위치와 글자
        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))

# 결과: 2개
# Match 1  13-15 → '4aa  13' 줄의 aa (이 줄은 12번에서 시작 → 4가 12번, aa가 13~14번)
# Match 2  33-35 → 'ababaa5' 줄의 aa (29번 시작 → a29 b30 a31 b32 a33 a34)
# 'aba'처럼 a 사이에 다른 글자가 있으면 안 됨. "aa"는 글자 그대로 딱 붙은 aa만


Match 1 was found at 13-15: aa
Match 2 was found at 33-35: aa


## Character classes

In [ ]:
print(test_str)
print("---------------------------------------")

# \w = 글자(영어, 숫자, 밑줄_), \W = 글자가 아닌 것(공백, 줄바꿈\n, @, ., -, 괄호 등)
# \w\W = "글자 하나 + 바로 뒤에 글자 아닌 것 하나" → 단어의 마지막 글자와 그 뒤 구분자를 찾는 셈
regex = r"\w\W"

matches = re.finditer(regex, test_str, re.MULTILINE)

# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))

# 결과 39개. 아 이래서 이렇게 나옴:
# - Match 1 'o ' : hello 뒤에 공백. Match 2는 화면에 'd'만 보이는데, 사실 'd' + 줄바꿈(\n)임 → \n도 \W라서 다음 줄로 넘어가 보이는 것
# - 'hello'의 'll'은 안 나옴: l 뒤가 또 글자(\w)라서
# - 'yaya---' → 'a-' 하나만: 그다음 '--'는 앞 글자가 -(\W)라서 \w\W가 안 됨
# - 이메일 'ikwak2@cau.ac.kr' → '2@', 'u.', 'c.', 'r'+\n : @와 . 이 \W
# - 'send me a message' → Match 27 'e ', Match 28 'a ' 가 붙어서 나옴: 한 글자 단어 a도 \w
# - 찾은 글자는 다시 안 씀(겹치지 않음). 'o '에서 공백을 썼으니 다음 검색은 'w'부터


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 4-6: o 
Match 2 was found at 10-12: d

Match 3 was found at 14-16: a 
Match 4 was found at 18-20: 3

Match 5 was found at 22-24: a 
Match 6 was found at 27-29: 4

Match 7 was found at 35-37: 5

Match 8 was found at 40-42: a-
Match 9 was found at 50-52: 2@
Match 10 was found at 54-56: u.
Match 11 was found at 57-59: c.
Match 12 was found at 60-62: r

Match 13 was found at 63-65: l.
Match 14 was found at 68-70: p.
Match 15 was found at 73-75: k@
Match 16 was found at 79-81: l.
Match 17 was found at 83-85: m

Match 18 was found at 86-88: 2-
Match 19 was found at 90-92: 1-
Match 20 was found at 95-97: 4

Match 21 was found at 99-101: 0-
Match 22 was found at 104-106: 4-
Match 23 was found at 109-111: 5

Match 24 was found at 121-123: 1

Matc

In [ ]:
print(test_str)
print("---------------------------------------")

# [ab] = a 또는 b 중 한 글자. 두 번 쓰면 → a/b로만 된 두 글자: aa, ab, ba, bb
regex = r"[ab][ab]"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 5개: aa(13-15), ab(20-22), ab(29-31), ab(31-33), aa(33-35)
# 아 이래서:
# - 'aba  b44' 에서 'ab'(20-22)만 나오고 'ba'(21-23)는 안 나옴 → 21번 b를 이미 'ab'에 써버렸기 때문 (겹치지 않음!)
#   다음 검색은 22번 a부터 → 뒤가 공백이라 실패
# - 'ababaa5' 는 ab / ab / aa 로 딱 세 조각
# - 'yaya', 'cau', 'kwak' 등은 a 옆이 a/b가 아니라서 없음


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 13-15: aa
Match 2 was found at 20-22: ab
Match 3 was found at 29-31: ab
Match 4 was found at 31-33: ab
Match 5 was found at 33-35: aa


In [ ]:
print(test_str)
print("---------------------------------------")


# [a-z] = 소문자 한 글자 (a부터 z까지 범위). 두 번 → 소문자 두 글자 연속
regex = r"[a-z][a-z]"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 42개. 아 이래서:
# - 'hello' → 'he', 'll' 그리고 'o'는 짝이 없어서 버려짐 (o 다음이 공백)
# - 'world' → 'wo', 'rl', 'd'는 버려짐 → 단어를 앞에서부터 두 글자씩 끊어 가져가는 모양
# - '4aa' → 'aa'만 (4는 숫자라 [a-z] 아님)
# - 'ikwak2' → 'ik', 'wa' 그리고 'k'는 뒤가 숫자 2라서 버려짐
# - [ab][ab]보다 범위가 넓어서 훨씬 많이 나옴


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 0-2: he
Match 2 was found at 2-4: ll
Match 3 was found at 6-8: wo
Match 4 was found at 8-10: rl
Match 5 was found at 13-15: aa
Match 6 was found at 20-22: ab
Match 7 was found at 29-31: ab
Match 8 was found at 31-33: ab
Match 9 was found at 33-35: aa
Match 10 was found at 37-39: ya
Match 11 was found at 39-41: ya
Match 12 was found at 45-47: ik
Match 13 was found at 47-49: wa
Match 14 was found at 52-54: ca
Match 15 was found at 56-58: ac
Match 16 was found at 59-61: kr
Match 17 was found at 62-64: il
Match 18 was found at 65-67: yo
Match 19 was found at 67-69: up
Match 20 was found at 70-72: kw
Match 21 was found at 72-74: ak
Match 22 was found at 75-77: gm
Match 23 was found at 77-79: ai
Match 24 was found at 81-83: co
Match 25 was fou

## Anchors

In [ ]:
print(test_str)
print("---------------------------------------")

# ^ = 시작 위치. ^\w\w = 시작하자마자 나오는 글자 두 개
# re.MULTILINE 옵션이 있어서 "문자열 전체의 시작"이 아니라 "각 줄의 시작"마다 적용됨
regex = r"^\w\w"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 12개 = 12줄 전부의 첫 두 글자: he, 4a, ab, ab, ya, ik, il, 02, 01, 01, he, ca
# 아 이래서:
# - '4a'도 나옴 → \w에는 숫자도 포함
# - 시작 위치가 0, 12, 20, 29, ... 로 위 요약표의 "줄 시작 위치"와 똑같음
# - re.MULTILINE을 빼면? → 'he' 하나만 나옴 (문자열 전체의 맨 처음만 ^로 봄)


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 0-2: he
Match 2 was found at 12-14: 4a
Match 3 was found at 20-22: ab
Match 4 was found at 29-31: ab
Match 5 was found at 37-39: ya
Match 6 was found at 45-47: ik
Match 7 was found at 62-64: il
Match 8 was found at 85-87: 02
Match 9 was found at 97-99: 01
Match 10 was found at 111-113: 01
Match 11 was found at 123-125: he
Match 12 was found at 168-170: ca


In [ ]:
print(test_str)
print("---------------------------------------")

# $ = 끝 위치. \w\w$ = 끝나기 직전 글자 두 개. MULTILINE이라 "각 줄의 끝"마다
regex = r"\w\w$"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 9개: ld, 13, 44, a5, kr, om, 34, 55, 11
# 아 이래서 12줄인데 9개뿐:
# - 'yaya---' → 끝이 '-' (\W)라서 실패
# - '...(name.kwak@cau.ac.kr)' 와 'call me (032 111 1424)' → 끝이 ')' 라서 실패
# - re.MULTILINE을 빼면? → 하나도 안 나옴 (전체 문자열의 마지막 줄이 ')'로 끝나서)


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 9-11: ld
Match 2 was found at 17-19: 13
Match 3 was found at 26-28: 44
Match 4 was found at 34-36: a5
Match 5 was found at 59-61: kr
Match 6 was found at 82-84: om
Match 7 was found at 94-96: 34
Match 8 was found at 108-110: 55
Match 9 was found at 120-122: 11


In [ ]:
print(test_str)
print("---------------------------------------")

# \d = 숫자 한 개. \b = 단어 경계(글자\w와 글자 아닌 것\W 사이, 또는 끝)
# \d\d\b = 숫자 두 개 + 바로 뒤가 경계 → 숫자 덩어리의 "마지막 두 자리"
regex = r"\d\d\b"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 12개: 13, 44, 02, 11, 34, 10, 34, 55, 11, 32, 11, 24
# 아 이래서:
# - '1234' 에서 '12'가 아니라 '34'가 나옴 → '12' 뒤는 '3'(글자)이라 경계가 아님 → 실패 → 한 칸 옆 '23' 도 실패 → '34' 뒤는 \n이라 경계 → 성공
# - '111' → '11'이 89-91 (뒤쪽 두 개). 앞쪽 '11'(88-90)은 뒤가 '1'이라 실패
# - '010' → '10', '01032321111' → 마지막 '11'
# - 'ikwak2', '4aa' 의 숫자는 한 자리뿐이라 없음


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 17-19: 13
Match 2 was found at 26-28: 44
Match 3 was found at 85-87: 02
Match 4 was found at 89-91: 11
Match 5 was found at 94-96: 34
Match 6 was found at 98-100: 10
Match 7 was found at 103-105: 34
Match 8 was found at 108-110: 55
Match 9 was found at 120-122: 11
Match 10 was found at 178-180: 32
Match 11 was found at 182-184: 11
Match 12 was found at 187-189: 24


In [ ]:
print(test_str)
print("---------------------------------------")

# \B = 경계가 "아닌" 곳 (\b의 반대)
# \d\d\B = 숫자 두 개 + 바로 뒤에 또 글자(숫자)가 이어짐 → 덩어리의 "끝이 아닌" 부분
regex = r"\d\d\B"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 13개. 아 이래서 바로 위(\b) 결과와 정반대로 나옴:
# - '1234' → 이번엔 '12' (뒤가 3이라 경계 아님 → 성공). 그다음 '34'는 뒤가 \n이라 실패
# - '111' → 앞쪽 '11'(88-90)
# - '01032321111' → '01', '03', '23', '21', '11' 다섯 조각 + 마지막 '1' 하나는 짝이 없어서 버려짐
# - '13', '44' 는 안 나옴 → 바로 뒤가 줄바꿈(경계)이라서
# 정리: 같은 숫자 덩어리를 \b는 "끝 두 자리", \B는 "그 앞쪽"으로 나눠 가짐


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 88-90: 11
Match 2 was found at 92-94: 12
Match 3 was found at 97-99: 01
Match 4 was found at 101-103: 32
Match 5 was found at 106-108: 55
Match 6 was found at 111-113: 01
Match 7 was found at 113-115: 03
Match 8 was found at 115-117: 23
Match 9 was found at 117-119: 21
Match 10 was found at 119-121: 11
Match 11 was found at 177-179: 03
Match 12 was found at 181-183: 11
Match 13 was found at 185-187: 14


## Quantifiers

In [ ]:
print(test_str)
print("---------------------------------------")

# + = 바로 앞이 1개 이상 반복. \d+ = 숫자가 이어진 덩어리 통째로
regex = r"\d+"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 15개: 4, 13, 44, 5, 2, 02, 111, 1234, 010, 3234, 5555, 01032321111, 032, 111, 1424
# 아 이래서:
# - 숫자가 끊기는 곳(하이픈, 공백, 글자)에서 덩어리가 나뉨 → '02-111-1234'는 세 덩어리
# - '01032321111'은 붙어 있어서 한 덩어리
# - greedy(최대한 길게)라서 '1234'를 '1','2'... 로 안 쪼개고 통째로


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 12-13: 4
Match 2 was found at 17-19: 13
Match 3 was found at 26-28: 44
Match 4 was found at 35-36: 5
Match 5 was found at 50-51: 2
Match 6 was found at 85-87: 02
Match 7 was found at 88-91: 111
Match 8 was found at 92-96: 1234
Match 9 was found at 97-100: 010
Match 10 was found at 101-105: 3234
Match 11 was found at 106-110: 5555
Match 12 was found at 111-122: 01032321111
Match 13 was found at 177-180: 032
Match 14 was found at 181-184: 111
Match 15 was found at 185-189: 1424


In [ ]:
print(test_str)
print("---------------------------------------")

# * = 바로 앞이 0개 이상. \d* = 숫자 0개 이상
regex = r"\d*"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과가 159개나 나옴! 대부분 'found at 0-0:' 처럼 시작=끝인 빈 문자열("")
# 아 이래서:
# - *는 "0개도 OK"라서, 숫자가 없는 위치에서도 "숫자 0개 찾았다!"로 성공해버림 → 글자 사이사이마다 빈 매칭
# - 숫자가 있는 곳은 덩어리 통째로 (12-13 '4', 17-19 '13', 26-28 '44' ...)
# - 덩어리 바로 다음 위치(13-13)에서도 빈 매칭이 하나 더 생김
# - 작은 예: re.findall(r'\d*', 'a12b') → ['', '12', '', '']
# 교훈: 뭔가를 "찾을" 때는 * 말고 + 를 써야 함


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 0-0: 
Match 2 was found at 1-1: 
Match 3 was found at 2-2: 
Match 4 was found at 3-3: 
Match 5 was found at 4-4: 
Match 6 was found at 5-5: 
Match 7 was found at 6-6: 
Match 8 was found at 7-7: 
Match 9 was found at 8-8: 
Match 10 was found at 9-9: 
Match 11 was found at 10-10: 
Match 12 was found at 11-11: 
Match 13 was found at 12-13: 4
Match 14 was found at 13-13: 
Match 15 was found at 14-14: 
Match 16 was found at 15-15: 
Match 17 was found at 16-16: 
Match 18 was found at 17-19: 13
Match 19 was found at 19-19: 
Match 20 was found at 20-20: 
Match 21 was found at 21-21: 
Match 22 was found at 22-22: 
Match 23 was found at 23-23: 
Match 24 was found at 24-24: 
Match 25 was found at 25-25: 
Match 26 was found at 26-28: 44
Match 27 was

In [ ]:
print(test_str)
print("---------------------------------------")

# ? = 바로 앞이 0개 또는 1개. \d? = 숫자 없거나 딱 1개
regex = r"\d?"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 192개 (\d*보다 더 많음). 아 이래서:
# - ?도 0개가 되니까 빈 매칭이 잔뜩 (\d*와 같은 이유)
# - 최대 1개라서 '13'을 통째로 못 잡고 '1'(17-18), '3'(18-19)으로 한 글자씩 → 매칭 개수가 더 늘어남
# - 작은 예: re.findall(r'\d?', 'a12b') → ['', '1', '2', '', '']


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 0-0: 
Match 2 was found at 1-1: 
Match 3 was found at 2-2: 
Match 4 was found at 3-3: 
Match 5 was found at 4-4: 
Match 6 was found at 5-5: 
Match 7 was found at 6-6: 
Match 8 was found at 7-7: 
Match 9 was found at 8-8: 
Match 10 was found at 9-9: 
Match 11 was found at 10-10: 
Match 12 was found at 11-11: 
Match 13 was found at 12-13: 4
Match 14 was found at 13-13: 
Match 15 was found at 14-14: 
Match 16 was found at 15-15: 
Match 17 was found at 16-16: 
Match 18 was found at 17-18: 1
Match 19 was found at 18-19: 3
Match 20 was found at 19-19: 
Match 21 was found at 20-20: 
Match 22 was found at 21-21: 
Match 23 was found at 22-22: 
Match 24 was found at 23-23: 
Match 25 was found at 24-24: 
Match 26 was found at 25-25: 
Match 27 was f

#### aa or there is one character between 'aa'

 aa 또는 aa 사이에 문자가 있는경우 찾기

In [ ]:
print(test_str)
print("---------------------------------------")

# a + [a-z]? + a → a로 시작, a로 끝, 가운데 소문자가 0개 또는 1개
#   = 'aa' (가운데 0개) 또는 'aba', 'aya' 같은 세 글자 (가운데 1개)
regex = r"a[a-z]?a"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 5개: aa, aba, aba, aa, aya
# 아 이래서:
# - 'ababaa5' (29번 시작) → 'aba'(29-32) 다음은 32번 b부터 다시 → 실패 → 33번 'aa'(33-35)
#   그래서 31-34의 'aba'는 안 나옴 (31번 a를 이미 썼으니까, 겹치지 않음)
# - 'yaya' → 'aya' (가운데 y)
# - 'kwak', 'cau', 'gmail' 은 a 다음 다음 자리에 a가 없어서 없음


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 13-15: aa
Match 2 was found at 20-23: aba
Match 3 was found at 29-32: aba
Match 4 was found at 33-35: aa
Match 5 was found at 38-41: aya


#### First 2-3 digits from phone numbers

전화번호 앞자리만 찾기

In [ ]:
print(test_str)
print("---------------------------------------")

# ^ (줄 시작) + \d{2,3} (숫자 2~3개) → 숫자 2~3개로 시작하는 줄의 앞자리
regex = r"^\d{2,3}"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 3개: '02', '010', '010'
# 아 이래서:
# - '02-111-1234' → 3번째 글자가 '-'라서 2개에서 멈춤
# - '01032321111' → 숫자가 계속 이어지지만 최대 3개라 '010'
# - {2,3}도 greedy(최대한 많이)라서 2개 될 때 멈추지 않고 3개까지 가져감
# - '4aa  13' 줄은 숫자로 시작하지만 1개뿐이라 실패
# - '(032 111 1424)'의 032는 줄 시작이 아니라서 안 나옴 (^ 덕분)


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 85-87: 02
Match 2 was found at 97-100: 010
Match 3 was found at 111-114: 010


## Groups & Lookaround

In [ ]:
print(test_str)
print("---------------------------------------")

# (\w\w) = 두 글자를 1번 그룹으로 잡음, \1 = 1번 그룹이 잡은 "바로 그 글자"가 한 번 더
# → 같은 두 글자가 두 번 반복되는 곳: abab, yaya, 1212 같은 것
regex = r"(\w\w)\1"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 5개: abab(그룹 ab), yaya(ya), 5555(55), 3232(32), 1111(11)
# 아 이래서:
# - 이번엔 Group 1 줄이 같이 출력됨 → 패턴에 괄호( )가 있어서 len(match.groups())가 1
# - \1은 "아무 두 글자"가 아니라 "1번 그룹이 실제로 잡은 글자" → 'ab' 다음에 또 'ab'여야 함
# - 'hello'의 'll'은 안 나옴 → 한 글자 반복이라서 (두 글자 단위가 반복돼야 함)
# - '01032321111' → '3232' 다음 위치부터 다시 → '1111'
# - 참고: re.findall로 하면 그룹만 돌려줘서 ['ab', 'ya', '55', '32', '11'] 이 나옴


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 29-33: abab
Group 1 found at 29-31: ab
Match 2 was found at 37-41: yaya
Group 1 found at 37-39: ya
Match 3 was found at 106-110: 5555
Group 1 found at 106-108: 55
Match 4 was found at 114-118: 3232
Group 1 found at 114-116: 32
Match 5 was found at 118-122: 1111
Group 1 found at 118-120: 11


In [ ]:
print(test_str)
print("---------------------------------------")

# (?: ) = 비캡처 그룹 → 묶기만 하고 그룹 번호를 안 매김 (꺼내지도 않음)
# (?:\w\w\w)@ = 글자 세 개 + @ → @ 바로 앞 세 글자
regex = r"(?:\w\w\w)@"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 3개: 'ak2@', 'wak@', 'wak@' (이메일 3개의 @ 앞 세 글자)
# 아 이래서:
# - 괄호가 있는데도 Group 줄이 안 나옴 → (?:)는 그룹 개수에 안 들어가서 len(match.groups())가 0
#   (\w\w\w)@ 로 썼다면 Group 1 'ak2' 줄이 같이 나왔을 것
# - 'ikwak2@' 전체가 아니라 'ak2@'만 → 글자 수를 정확히 3개로 정했으니까
# - 이 셀은 (?:) 시연용이라 사실 \w\w\w@ 와 결과가 똑같음. 진짜 쓰임은 맨 마지막 마스킹 셀에서 나옴


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 48-52: ak2@
Match 2 was found at 71-75: wak@
Match 3 was found at 153-157: wak@


## Practice

### Search for e-mail addresses

이메일 주소 찾기

In [ ]:
print(test_str)
print("---------------------------------------")

# 이메일 패턴을 조각내서 보면:
#   [\w\.-]+   → @ 앞부분: 글자, 점(.), 하이픈(-)이 1개 이상 (예: il.youp.kwak)
#   @          → 골뱅이 그대로
#   [\w\.]+    → 도메인: 글자와 점이 1개 이상 (예: cau.ac)
#   .          → 아무 글자 1개 (역슬래시가 없어서 "진짜 점"이 아니라 "아무 글자")
#   \w{2,3}    → 끝부분 글자 2~3개 (kr, com)
# 대괄호 안에서는 \. 과 . 둘 다 진짜 점, - 는 맨 끝에 있어서 범위가 아니라 진짜 하이픈
regex = r"[\w\.-]+@[\w\.]+.\w{2,3}"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 3개: ikwak2@cau.ac.kr / il.youp.kwak@gmail.com / name.kwak@cau.ac.kr
# 아 이래서:
# - 세 번째는 'message(name.kwak...' 에서 'name.kwak'부터 시작 → '(' 가 [\w\.-] 에 없어서 거기서 끊김
#   끝의 ')' 도 [\w\.] 에 없어서 안 들어감
# - 'cau.ac.kr' 을 [\w\.]+ 가 처음엔 끝까지 다 먹어버림(greedy) → 그러면 뒤의 . 과 \w{2,3} 이 먹을 게 없어서 실패
#   → 한 글자씩 되돌려줌(backtracking) → [\w\.]+ = 'cau.ac', . = '.', \w{2,3} = 'kr' 에서 성공
# - 주의: 끝의 . 이 "아무 글자"라서 'a@bxcom' 같은 가짜 주소도 잡힘. 정확히 하려면 \. 으로 써야 함
#   (이 test_str에는 그런 경우가 없어서 결과는 똑같음)


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 45-61: ikwak2@cau.ac.kr
Match 2 was found at 62-84: il.youp.kwak@gmail.com
Match 3 was found at 147-166: name.kwak@cau.ac.kr


### Search for phone numbers

전화번호 찾기

In [ ]:
print(test_str)
print("---------------------------------------")

# \d{2,3}  → 앞자리 숫자 2~3개 (02, 010, 032)
# [- ]?    → 하이픈 또는 공백이 0개 또는 1개 (있어도 되고 없어도 됨). 대괄호 맨 앞의 - 는 범위가 아니라 진짜 하이픈
# \d{3,4}  → 가운데 숫자 3~4개
# [- ]?    → 또 구분자 있어도/없어도
# \d{4}    → 끝 숫자 4개
regex = r"\d{2,3}[- ]?\d{3,4}[- ]?\d{4}"

matches = re.finditer(regex, test_str, re.MULTILINE)
# (아래 for문은 4번째 코드 셀과 똑같음: 찾은 것마다 번호, 위치, 글자를 출력. 괄호 그룹이 있으면 Group 줄도)
for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
# 결과 4개: 02-111-1234 / 010-3234-5555 / 01032321111 / 032 111 1424
# 아 이래서:
# - 하이픈(02-111-1234), 붙여쓰기(01032321111), 공백(032 111 1424) 세 가지 모양이 다 잡힘 → [- ]? 덕분
# - '01032321111' → 앞 '010'(3개), 구분자 없음, 가운데 '3232'(4개), 끝 '1111'(4개)
# - '13', '44', '2' 같은 짧은 숫자는 길이가 모자라서 안 잡힘


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-111-1234
010-3234-5555
01032321111
hello send me a message(name.kwak@cau.ac.kr)
call me (032 111 1424)

---------------------------------------
Match 1 was found at 85-96: 02-111-1234
Match 2 was found at 97-110: 010-3234-5555
Match 3 was found at 111-122: 01032321111
Match 4 was found at 177-189: 032 111 1424


### masking middle phone numbers

전화번호 가운데 숫자들 감추기

In [ ]:
import re

# 위 전화번호 패턴에 괄호를 씌운 것:
#   (\d{2,3})     → 1번 그룹 = 앞자리
#   (?:\d{3,4})   → 비캡처 = 가운데 (묶기만, 번호 없음)
#   (\d{4})       → 2번 그룹 = 끝 4자리  ← 가운데가 (?:)라서 이게 3번이 아니라 2번이 됨!
regex = r"(\d{2,3})[- ]?(?:\d{3,4})[- ]?(\d{4})"

# 바꿀 문자열: \1(앞자리) + -****- + \2(끝 4자리)
# 일반 문자열("")이라서 역슬래시를 두 번(\\) 씀. r"\1-****-\2" 와 같은 뜻
subst = "\\1-****-\\2"

# You can manually specify the number of replacements by changing the 4th argument
# re.sub(패턴, 바꿀것, 문자열, 0, re.MULTILINE)
#   4번째 0 = count(몇 개 바꿀지). 0이면 전부
#   5번째 = flags(옵션)
# ※ 파이썬 3.13부터는 count와 flags를 이렇게 순서로만 주면 경고가 뜸
#    → re.sub(regex, subst, test_str, count=0, flags=re.MULTILINE) 처럼 이름을 붙여 쓰면 됨
result = re.sub(regex, subst, test_str, 0, re.MULTILINE)

# result가 빈 문자열이 아니면 출력
if result:
    print (result)
# 결과: 02-****-1234 / 010-****-5555 / 010-****-1111 / (032-****-1424)
# 아 이래서:
# - 원래 공백으로 구분돼 있던 '032 111 1424'도 '032-****-1424'로 바뀜 → 바꿀 문자열에 - 를 직접 넣었으니까
# - 붙여쓰기 '01032321111'도 '010-****-1111' 로 하이픈이 생김
# - 가운데를 (?:)가 아니라 (\d{3,4})로 캡처했다면? → 끝 4자리가 3번 그룹이 되고 \2는 가운데를 가리킴
#   → '02-****-111', '010-****-3234' 처럼 틀린 결과가 나옴. 이래서 가운데를 (?:)로 묶은 것!


hello world
4aa  13
aba  b44
ababaa5
yaya---
ikwak2@cau.ac.kr
il.youp.kwak@gmail.com
02-****-1234
010-****-5555
010-****-1111
hello send me a message(name.kwak@cau.ac.kr)
call me (032-****-1424)

